# 05 — AWS SageMaker Model Training & Serverless Endpoint Deployment

**Project:** THERMOX AI — Building Energy Efficiency Prediction  
**Track:** DEPI — AWS Machine Learning Track | **Team:** CloudXAi (`NHA-5-165`)

---

### Workflow Overview
1. **Step 1 — Connect to S3:** Load `train.csv` (614 rows) and `test.csv` (154 rows) from `s3://sagemaker-us-east-1-721093132276/thermox-ai/data/processed/`.
2. **Step 2 — Write `sagemaker_src/train.py`:** Project feature engineering (`Wall_Surface_Ratio`, `Glazing_x_Height`), preprocessing, and training of all **10 models**.
3. **Step 3 — Train, Package `model.tar.gz` & Upload to S3:** Runs `train.py` directly on the SageMaker Notebook CPU (bypassing new-account `ml.m5.large` training job quota blocks), packs `model.tar.gz`, and uploads it to `s3://sagemaker-us-east-1-721093132276/thermox-ai/models/model.tar.gz`.
4. **Step 4 — Write `sagemaker_src/inference.py`:** Endpoint serving handler (`model_fn`, `input_fn`, `predict_fn`, `output_fn`) matching FastAPI's JSON contract.
5. **Step 5 — Deploy Serverless Endpoint (`thermox-ai-endpoint`):** Deploys `model.tar.gz` with `ServerlessInferenceConfig` (**$0.00/hr** idle cost).
6. **Step 6 — Test Live Endpoint:** Invokes `thermox-ai-endpoint` via `boto3` (`sagemaker-runtime`).

## Step 1: Connect to S3 & Inspect `train.csv` and `test.csv`

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib", "xgboost"], check=True)

import os
import boto3
import pandas as pd
import sagemaker

os.makedirs("sagemaker_src", exist_ok=True)
os.makedirs("local_data", exist_ok=True)
os.makedirs("model_out", exist_ok=True)

REGION = "us-east-1"
BUCKET = "sagemaker-us-east-1-721093132276"
PREFIX = "thermox-ai"
ENDPOINT_NAME = "thermox-ai-endpoint"

role = sagemaker.get_execution_role()
s3_data_uri = f"s3://{BUCKET}/{PREFIX}/data/processed"

train_df = pd.read_csv(f"{s3_data_uri}/train.csv")
test_df = pd.read_csv(f"{s3_data_uri}/test.csv")
train_df.to_csv("local_data/train.csv", index=False)
test_df.to_csv("local_data/test.csv", index=False)

print("IAM Role   :", role)
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
train_df.head(3)

## Step 2: Create `sagemaker_src/train.py` (Extracted Project Training Phase)

In [ ]:
%%writefile sagemaker_src/train.py
import json
import math
import os
import shutil
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

RAW_FEATURES = [
    "Relative_Compactness", "Surface_Area", "Wall_Area", "Roof_Area",
    "Overall_Height", "Orientation", "Glazing_Area", "Glazing_Area_Distribution"
]
ENGINEERED_FEATURES = ["Wall_Surface_Ratio", "Glazing_x_Height"]
ALL_FEATURES = RAW_FEATURES + ENGINEERED_FEATURES
CAT_COLS = ["Orientation", "Glazing_Area_Distribution"]
NUM_COLS = [c for c in ALL_FEATURES if c not in CAT_COLS]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df[RAW_FEATURES].copy()
    out["Wall_Surface_Ratio"] = out["Wall_Area"] / out["Surface_Area"]
    out["Glazing_x_Height"] = out["Glazing_Area"] * out["Overall_Height"]
    return out[ALL_FEATURES]


def fit_bundle_from_df(train_df: pd.DataFrame, test_df: pd.DataFrame = None) -> dict:
    X_train_eng = add_features(train_df)
    prep_linear = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUM_COLS),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ],
        sparse_threshold=0.0,
    )
    X_train_lin = prep_linear.fit_transform(X_train_eng)
    X_train_tree = X_train_eng.values

    if test_df is not None:
        X_test_eng = add_features(test_df)
        X_test_lin = prep_linear.transform(X_test_eng)
        X_test_tree = X_test_eng.values

    def get_algorithms():
        return {
            "Linear Regression": LinearRegression(),
            "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=42),
            "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
            "Gradient Boosting": GradientBoostingRegressor(
                n_estimators=300, learning_rate=0.08, max_depth=4, random_state=42
            ),
            "XGBoost": XGBRegressor(
                n_estimators=400, learning_rate=0.05, max_depth=4,
                subsample=0.85, colsample_bytree=0.9, random_state=42, n_jobs=1
            ),
        }

    models = {"heating": {}, "cooling": {}}
    benchmarks = []

    for target_col, key in [("Heating_Load", "heating"), ("Cooling_Load", "cooling")]:
        y_train = train_df[target_col].values
        for algo_name, model in get_algorithms().items():
            X_tr = X_train_lin if algo_name == "Linear Regression" else X_train_tree
            model.fit(X_tr, y_train)
            models[key][algo_name] = model

            if test_df is not None:
                y_test = test_df[target_col].values
                X_te = X_test_lin if algo_name == "Linear Regression" else X_test_tree
                preds = model.predict(X_te)
                mae = round(float(mean_absolute_error(y_test, preds)), 4)
                rmse = round(float(math.sqrt(mean_squared_error(y_test, preds))), 4)
                r2 = round(float(r2_score(y_test, preds)), 4)
                benchmarks.append({"algorithm": algo_name, "target": target_col, "mae": mae, "rmse": rmse, "r2": r2})
                print(f"{algo_name:18s} | {target_col:12s} | MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f}")

    return {"models": models, "preprocessors": {"linear": prep_linear}, "benchmarks": benchmarks}


def main():
    data_dir = os.environ.get("SM_CHANNEL_DATA", "local_data")
    model_dir = os.environ.get("SM_MODEL_DIR", "model_out")
    os.makedirs(model_dir, exist_ok=True)

    train_path = os.path.join(data_dir, "train.csv")
    test_path = os.path.join(data_dir, "test.csv")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path) if os.path.exists(test_path) else None

    bundle = fit_bundle_from_df(train_df, test_df)
    joblib.dump(bundle, os.path.join(model_dir, "model_bundle.joblib"), compress=3)
    shutil.copy2(train_path, os.path.join(model_dir, "train.csv"))
    with open(os.path.join(model_dir, "metrics.json"), "w") as f:
        json.dump(bundle["benchmarks"], f, indent=2)
    print("Saved model_bundle.joblib successfully!")


if __name__ == "__main__":
    main()


## Step 3: Train All 10 Models, Build `model.tar.gz`, & Upload to S3
New AWS Free Tier accounts default to `0` quota for separate `ml.m5.large` Training Job clusters. Running `train.py` directly on your SageMaker Notebook CPU trains all 10 models in ~3 seconds, packages `model.tar.gz`, and uploads it straight to your S3 `models/` folder!

In [ ]:
import subprocess
import sys
import tarfile

# 1. Ensure required ML packages exist in this exact kernel & run train.py
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib", "xgboost"], check=True)
os.makedirs("local_data", exist_ok=True)
os.makedirs("model_out", exist_ok=True)
train_df.to_csv("local_data/train.csv", index=False)
test_df.to_csv("local_data/test.csv", index=False)

subprocess.run([sys.executable, "sagemaker_src/train.py"], check=True)

# 2. Package model_out/ into model.tar.gz
with tarfile.open("model.tar.gz", "w:gz") as tar:
    for fname in os.listdir("model_out"):
        tar.add(os.path.join("model_out", fname), arcname=fname)

# 3. Upload model.tar.gz to your S3 bucket
s3_model_key = f"{PREFIX}/models/model.tar.gz"
boto3.client("s3", region_name=REGION).upload_file("model.tar.gz", BUCKET, s3_model_key)
model_data_uri = f"s3://{BUCKET}/{s3_model_key}"
print("\nUploaded model.tar.gz to S3:", model_data_uri)

## Step 4: Create `sagemaker_src/inference.py` (Endpoint Serving Code Only)
Loads `model_bundle.joblib` into RAM (and automatically rebuilds in ~1s from `train.csv` if the container's `scikit-learn` minor version differs from the notebook kernel, guaranteeing zero pickle version errors!).

In [ ]:
%%writefile sagemaker_src/inference.py
import json
import os
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

RAW_FEATURES = [
    "Relative_Compactness", "Surface_Area", "Wall_Area", "Roof_Area",
    "Overall_Height", "Orientation", "Glazing_Area", "Glazing_Area_Distribution"
]
ALL_FEATURES = RAW_FEATURES + ["Wall_Surface_Ratio", "Glazing_x_Height"]
CAT_COLS = ["Orientation", "Glazing_Area_Distribution"]
NUM_COLS = [c for c in ALL_FEATURES if c not in CAT_COLS]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df[RAW_FEATURES].copy()
    out["Wall_Surface_Ratio"] = out["Wall_Area"] / out["Surface_Area"]
    out["Glazing_x_Height"] = out["Glazing_Area"] * out["Overall_Height"]
    return out[ALL_FEATURES]


def _fit_native(train_csv_path: str) -> dict:
    train_df = pd.read_csv(train_csv_path)
    X_eng = add_features(train_df)
    prep_linear = ColumnTransformer(
        [("num", StandardScaler(), NUM_COLS), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS)],
        sparse_threshold=0.0,
    )
    X_lin = prep_linear.fit_transform(X_eng)
    X_tree = X_eng.values
    models = {"heating": {}, "cooling": {}}
    for target_col, key in [("Heating_Load", "heating"), ("Cooling_Load", "cooling")]:
        y = train_df[target_col].values
        algos = {
            "Linear Regression": LinearRegression(),
            "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=42),
            "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
            "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.08, max_depth=4, random_state=42),
            "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.85, colsample_bytree=0.9, random_state=42, n_jobs=1),
        }
        for name, m in algos.items():
            m.fit(X_lin if name == "Linear Regression" else X_tree, y)
            models[key][name] = m
    return {"models": models, "preprocessors": {"linear": prep_linear}}


def _predict_one(features: dict, algorithm: str, bundle: dict) -> dict:
    algo = algorithm if algorithm in bundle["models"]["heating"] else "XGBoost"
    df_eng = add_features(pd.DataFrame([features]))
    X_in = bundle["preprocessors"]["linear"].transform(df_eng) if algo == "Linear Regression" else df_eng.values

    heat = round(float(bundle["models"]["heating"][algo].predict(X_in)[0]), 2)
    cool = round(float(bundle["models"]["cooling"][algo].predict(X_in)[0]), 2)
    return {
        "heating_load_kwh": heat,
        "cooling_load_kwh": cool,
        "total_load_kwh": round(heat + cool, 2),
        "algorithm_used": algo,
        "inference_source": "aws-sagemaker",
        "engineered_features": {
            "Wall_Surface_Ratio": round(float(features["Wall_Area"]) / float(features["Surface_Area"]), 4),
            "Glazing_x_Height": round(float(features["Glazing_Area"]) * float(features["Overall_Height"]), 4),
        },
    }


def model_fn(model_dir: str):
    """Loads model_bundle.joblib (or rebuilds natively if sklearn Cython version differs)."""
    try:
        bundle = joblib.load(os.path.join(model_dir, "model_bundle.joblib"))
        # Smoke-test 1 row to verify sklearn/xgboost C-struct compatibility
        _predict_one(
            {"Relative_Compactness": 0.98, "Surface_Area": 514.5, "Wall_Area": 294.0, "Roof_Area": 110.25,
             "Overall_Height": 7.0, "Orientation": 2, "Glazing_Area": 0.10, "Glazing_Area_Distribution": 1},
            "Random Forest",
            bundle,
        )
        return bundle
    except Exception as exc:
        print(f"[model_fn] Rebuilding native container models due to version difference ({exc})...")
        return _fit_native(os.path.join(model_dir, "train.csv"))


def input_fn(request_body: str, request_content_type: str = "application/json"):
    return json.loads(request_body)


def predict_fn(data: dict, bundle: dict) -> dict:
    if data.get("action") == "optimize":
        param = data.get("optimize_param", "Glazing_Area")
        base = dict(data["base_features"])
        algo = data.get("algorithm", "XGBoost")
        candidates = (
            [0.0, 0.10, 0.25, 0.40] if param == "Glazing_Area"
            else [2, 3, 4, 5] if param == "Orientation"
            else [0.64, 0.74, 0.82, 0.90, 0.98]
        )
        evals, best_load, best_val = [], float("inf"), candidates[0]
        for val in candidates:
            row = {**base, param: val}
            res = _predict_one(row, algo, bundle)
            if res["total_load_kwh"] < best_load:
                best_load, best_val = res["total_load_kwh"], val
            evals.append({
                "param_value": val,
                "heating_load": res["heating_load_kwh"],
                "cooling_load": res["cooling_load_kwh"],
                "total_load": res["total_load_kwh"],
            })
        return {
            "parameter_optimized": param,
            "optimal_value": best_val,
            "lowest_total_load": best_load,
            "evaluations": evals,
            "inference_source": "aws-sagemaker",
        }

    features = data.get("features", data)
    algorithm = data.get("algorithm", "XGBoost")
    return _predict_one(features, algorithm, bundle)


def output_fn(prediction: dict, accept: str = "application/json"):
    return json.dumps(prediction), "application/json"


## Step 5: Deploy the Serverless Endpoint (`thermox-ai-endpoint`)
Uses `ServerlessInferenceConfig` so the endpoint scales to **$0.00/hour** whenever there are no requests.

In [ ]:
from sagemaker.xgboost import XGBoostModel
from sagemaker.serverless import ServerlessInferenceConfig

sm_client = boto3.client("sagemaker", region_name=REGION)
for fn, arg in [
    (sm_client.delete_endpoint, {"EndpointName": ENDPOINT_NAME}),
    (sm_client.delete_endpoint_config, {"EndpointConfigName": ENDPOINT_NAME}),
]:
    try:
        fn(**arg)
    except Exception:
        pass

sm_model = XGBoostModel(
    model_data=model_data_uri,
    role=role,
    entry_point="inference.py",
    source_dir="sagemaker_src",
    framework_version="1.7-1",
)

predictor = sm_model.deploy(
    endpoint_name=ENDPOINT_NAME,
    serverless_inference_config=ServerlessInferenceConfig(
        memory_size_in_mb=2048,
        max_concurrency=5,
    ),
)

print(f"Endpoint '{ENDPOINT_NAME}' is LIVE and InService!")

## Step 6: Test the Endpoint (Exact Same `boto3` Call Used by FastAPI)

In [ ]:
import json

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

sample_building = {
    "Relative_Compactness": 0.98,
    "Surface_Area": 514.5,
    "Wall_Area": 294.0,
    "Roof_Area": 110.25,
    "Overall_Height": 7.0,
    "Orientation": 2,
    "Glazing_Area": 0.10,
    "Glazing_Area_Distribution": 1,
}

# 1. Test Single Prediction (/api/predict contract)
resp = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps({"action": "predict", "algorithm": "XGBoost", "features": sample_building}),
)
print("Predict Response:", json.loads(resp["Body"].read().decode()))

# 2. Test Design Optimization (/api/optimize contract)
opt_resp = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps({"action": "optimize", "optimize_param": "Glazing_Area", "base_features": sample_building}),
)
print("Optimize Response:", json.loads(opt_resp["Body"].read().decode()))